# HotpotQA v2.1 — robust candidate-ID filtering
Reuse a construction ZIP; compare dense, Entity, Entity–Event and Full-KG with local Qwen3.5-2B.
The filter returns integer candidate IDs, never rewritten triples. Invalid/incomplete responses are retried at most twice. By default, exhausted **format** retries use a clearly logged dense-passage fallback for that question. Connection/authentication/server failures still stop.
This is a small-model adaptation, not an exact paper reproduction.

**Use a NEW RUN_ROOT.** Do not mix v2.1 results with old text-filter or no-filter runs. Existing graph ZIP can be reused without rebuilding. Three questions remain a smoke test.


In [ ]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path
from datetime import datetime
import requests

gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout)
if gpu.returncode != 0:
    raise RuntimeError('Select Runtime > Change runtime type > GPU before continuing')
from google.colab import drive, files
drive.mount('/content/drive')

RUN_ROOT = Path('/content/drive/MyDrive/AutoSchemaKG/hotpotqa_v2_id_filter_smoke')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
INPUT_ZIP = RUN_ROOT / 'construction.zip'
PREVIOUS_INPUT = Path('/content/drive/MyDrive/AutoSchemaKG/hotpotqa_v2_smoke/construction.zip')
OUTPUT_DIR = RUN_ROOT / 'benchmark'
MODEL_ID = 'Qwen/Qwen3.5-2B'
EMBEDDING_MODEL = 'sentence-transformers/multi-qa-MiniLM-L6-cos-v1'
CODE_REF = 'codex/research-concept-retrieval'  # Must contain this update on GitHub.
PORT = 8000
CONTEXT_LENGTH = 4096
FILTER_MAX_ATTEMPTS = 2
FILTER_FAILURE_POLICY = 'dense'  # 'error' for strict experiments; new output root when changed.
print('Persistent experiment directory:', RUN_ROOT)
print('Filter policy:', FILTER_FAILURE_POLICY, 'attempts:', FILTER_MAX_ATTEMPTS)


## 1. Restore input without touching old outputs
If PREVIOUS_INPUT exists, copy that construction ZIP into the new run folder. Otherwise upload the construction/evaluated ZIP, not content.zip. Existing new-run input is never overwritten.


In [ ]:
if not INPUT_ZIP.exists():
    if PREVIOUS_INPUT.is_file() and PREVIOUS_INPUT.resolve() != INPUT_ZIP.resolve():
        shutil.copy2(PREVIOUS_INPUT, INPUT_ZIP)
        print('Reused graph ZIP:', PREVIOUS_INPUT)
    else:
        uploaded = files.upload()
        candidates = [Path(name).resolve() for name in uploaded if name.lower().endswith('.zip')]
        if len(candidates) != 1:
            raise ValueError('Upload exactly one construction ZIP')
        shutil.copy2(candidates[0], INPUT_ZIP)
print('Using:', INPUT_ZIP, 'bytes:', INPUT_ZIP.stat().st_size)


## 2. Clone the updated code and pin revisions
The update must be pushed to CODE_REF first. Fresh runs fetch that branch; resume restores the saved commit, not a newer branch head.
Old revisions.json files cannot be reused: create a new RUN_ROOT. Never delete old checkpoints to bypass configuration validation.


In [ ]:
REPO_DIR = Path('/content/SmallScaledAutoSchemaKG_v2_id_filter')
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', CODE_REF,
        'https://github.com/phuongth05/SmallScaledAutoSchemaKG.git', str(REPO_DIR)], check=True)
status = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip()
if status:
    raise RuntimeError('Clone has local edits; preserve them before switching revision')
revision_file = RUN_ROOT / 'revisions.json'
if revision_file.exists():
    revisions = json.loads(revision_file.read_text())
    if revisions.get('filter_protocol') != 'candidate_ids_v1':
        raise ValueError('Old filter run: use a NEW RUN_ROOT, keep old outputs unchanged')
    if revisions['model'] != MODEL_ID or revisions['embedding'] != EMBEDDING_MODEL:
        raise ValueError('Model changed: use a NEW RUN_ROOT')
    subprocess.run(['git', 'checkout', '--detach', revisions['git_commit']], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO_DIR, check=True)
    helper = REPO_DIR / 'scripts/colab_v2_utils.py'
    if not helper.exists() or 'candidate_ids_v1' not in helper.read_text():
        raise RuntimeError('This branch does not contain the ID-filter fix yet; push the update first')
    def model_revision(model):
        response = requests.get(f'https://huggingface.co/api/models/{model}', timeout=30)
        response.raise_for_status()
        return response.json()['sha']
    revisions = {
        'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip(),
        'filter_protocol': 'candidate_ids_v1',
        'model': MODEL_ID, 'model_revision': model_revision(MODEL_ID),
        'embedding': EMBEDDING_MODEL, 'embedding_revision': model_revision(EMBEDDING_MODEL),
    }
    revision_file.write_text(json.dumps(revisions, indent=2))
os.chdir(REPO_DIR)
print(json.dumps(revisions, indent=2))


## 3. Install isolated environments
QA embeddings run on CPU; only vLLM uses GPU. Separate environments avoid the Torch/TorchAudio CUDA mismatch encountered in the earlier notebook. No changes to Colab's preinstalled TorchAudio/TorchVision are required. First installation can take several minutes. Locks are saved to Drive for the next runtime.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
QA_ENV = '/content/autoschema_qa_id_filter_env'
VLLM_ENV = '/content/autoschema_vllm_id_filter_env'
QA_PY = QA_ENV + '/bin/python'
VLLM_PY = VLLM_ENV + '/bin/python'
for env in (QA_ENV, VLLM_ENV):
    if not Path(env, 'bin/python').exists():
        subprocess.run(['uv', 'venv', '--python', '3.12', env], check=True)
qa_lock = RUN_ROOT / 'qa_requirements.lock.txt'
subprocess.run(['uv', 'pip', 'install', '--python', QA_PY, '--torch-backend=cpu', '-r',
                str(qa_lock) if qa_lock.exists() else 'requirements-hotpotqa-v2.txt'], check=True)
if not qa_lock.exists():
    qa_lock.write_text(subprocess.check_output(['uv', 'pip', 'freeze', '--python', QA_PY], text=True))
vllm_lock = RUN_ROOT / 'vllm_requirements.lock.txt'
vllm_args = ['-r', str(vllm_lock)] if vllm_lock.exists() else ['--pre', 'vllm']
subprocess.run(['uv', 'pip', 'install', '--python', VLLM_PY, '--torch-backend=auto'] + vllm_args, check=True)
if not vllm_lock.exists():
    vllm_lock.write_text(subprocess.check_output(['uv', 'pip', 'freeze', '--python', VLLM_PY], text=True))
subprocess.run([QA_PY, '-u', 'scripts/run_hotpotqa_benchmark.py', str(INPUT_ZIP), '--inspect-only'], check=True)

## 4. Start local Qwen
Wait for `/v1/models` to be ready. If the server exits, the cell prints its log tail. Keep this runtime alive during the benchmark. After a runtime reset, restart this server; checkpoints remain in Drive.

In [ ]:
LOG_PATH = RUN_ROOT / 'qwen_vllm.log'
def ready():
    try:
        response = requests.get(f'http://127.0.0.1:{PORT}/v1/models', timeout=5)
        if response.ok:
            models = response.json()['data']
            if not any(m['id'] == MODEL_ID for m in models):
                raise RuntimeError('Port is occupied by another model; change PORT')
            return True
    except requests.RequestException:
        return False
    return False
if not ready():
    server_log = open(LOG_PATH, 'a', encoding='utf-8')
    server = subprocess.Popen([VLLM_ENV + '/bin/vllm', 'serve', MODEL_ID,
        '--revision', revisions['model_revision'], '--host', '127.0.0.1', '--port', str(PORT),
        '--dtype', 'half', '--max-model-len', str(CONTEXT_LENGTH), '--max-num-seqs', '1',
        '--gpu-memory-utilization', '0.80', '--language-model-only'],
        stdout=server_log, stderr=subprocess.STDOUT)
    started = time.monotonic()
    while time.monotonic() - started < 1200:
        if server.poll() is not None:
            server_log.flush()
            print(LOG_PATH.read_text(errors='replace')[-12000:])
            raise RuntimeError('vLLM stopped; inspect log above')
        if ready():
            break
        print(f'Waiting for Qwen: {time.monotonic() - started:.0f}s', flush=True)
        time.sleep(10)
    else:
        raise TimeoutError(f'Server not ready. Inspect {LOG_PATH}; do not start another copy')
print('Local Qwen is ready')

## 5. Run / resume with live logs
Use candidate-ID filtering, not --no-filter-edges. The chosen fallback policy is stored in run_config.json.
A format-error fallback affects that question only and is counted in summary diagnostics. Strict policy 'error' stops after retries and saves raw responses.
If the child fails, its actual error tail is shown here and the full log stays on Drive. Server failures are not silently converted to dense retrieval.


In [ ]:
sys.path.insert(0, str(REPO_DIR / 'scripts'))
from colab_v2_utils import normalize_base_url, run_logged

# Build inside this cell so an older Markdown-contaminated command cannot overwrite it.
BASE_URL = normalize_base_url('http' + '://' + '127.0.0.1:' + str(int(PORT)) + '/v1')
print('Base URL:', repr(BASE_URL), flush=True)

command = [QA_PY, '-u', 'scripts/run_hotpotqa_benchmark.py', str(INPUT_ZIP),
    '--output-dir', str(OUTPUT_DIR), '--variants', 'dense', 'entity', 'entity_event', 'full',
    '--model', MODEL_ID, '--model-revision', revisions['model_revision'],
    '--base-url', BASE_URL,
    '--embedding-model', EMBEDDING_MODEL, '--embedding-revision', revisions['embedding_revision'],
    '--embedding-device', 'cpu', '--top-edges', '30', '--top-passages', '5',
    '--ppr-alpha', '0.9', '--passage-weight', '0.9', '--context-length', str(CONTEXT_LENGTH),
    '--filter-max-attempts', str(FILTER_MAX_ATTEMPTS),
    '--filter-failure-policy', FILTER_FAILURE_POLICY]

LOG_PATH = RUN_ROOT / 'logs' / ('benchmark_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f') + '.log')
run_logged(command, LOG_PATH, cwd=REPO_DIR)


## 6. Inspect results and archive
Each method reports completed/expected questions and filter-error fallback/retry counts.
Do not present a run with many fallbacks as successful graph retrieval. An empty valid selection and an invalid-output fallback have separate trace reasons.
A historical last_error.json may remain after recovery; inspect its timestamp and current completion counts.


In [ ]:
summary_path = OUTPUT_DIR / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
    for method, data in summary['methods'].items():
        print(method, 'completed:', data['completed'], 'complete:', data['complete'])
        print('Filter diagnostics:', data.get('diagnostics', {}))
archive = shutil.make_archive('/content/autoschemakg_hotpotqa_v2_id_filter', 'zip', RUN_ROOT)
print('Checkpoint directory remains on Drive:', RUN_ROOT)
files.download(archive)
